<a href="https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
os.chdir("/content")
REPO_URL = "https://github.com/Santosh-S321/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


## 1. My rule and its reason codes

**Dataset note:** using the starter CSV, not the warehouse — it ships `days_since_last_update`,
`ctr`, and `avg_position` directly, matching FlyRank's real documented reason codes
(`stale_visible_page`, `low_ctr_visible_page`) without needing extra joins.

**Signal 1 — staleness (behind FlyRank's `stale_visible_page` refresh flag):** pages
not updated in 180+ days should show a higher decline rate than fresher pages.

**Signal 2 — CTR vs. position (behind FlyRank's `low_ctr_visible_page` CTR-fix logic):**
CTR should drop as position tier worsens.

**My rule, in plain words:** a page is worth reviewing for refresh if it hasn't been
updated in 180+ days AND it's still getting meaningful traffic (500+ impressions in 90
days) — stale content that's still visible is exactly where a refresh pays off.

**Score:** `score = is_stale * is_visible * impressions_90d`
**Reason code:** `stale_visible_page`
**Action label:** `review_for_refresh`

In [2]:
df["staleness_bucket"] = pd.cut(df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10_000], labels=["<90d", "90-180d", "180-365d", "365d+"])
bucket1 = df.groupby("staleness_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
print("Signal 1 — staleness vs decline rate:")
print(bucket1)

bucket2 = df.groupby("position_tier", observed=True)["ctr"].agg(["mean", "count"])
print("\nSignal 2 — CTR by position tier:")
print(bucket2.sort_values("mean", ascending=False))

Signal 1 — staleness vs decline rate:
                      mean  count
staleness_bucket                 
<90d              0.512031  20655
90-180d           0.611057   9171
180-365d          0.467456    169
365d+             0.600000      5

Signal 2 — CTR by position tier:
                   mean  count
position_tier                 
top_3          1.483611   2321
page_1         0.652467  11814
striking       0.323239   7304
page_3_5       0.222484   7242
deep           0.150212   1319


**Signal 1 verdict: MIXED.** Decline rate rises from <90d (0.512) to 90-180d (0.611),
consistent with the staleness hypothesis — but then drops at 180-365d (0.467), the
opposite direction, before 365d+ shows 0.600 on just n=5 rows (too small to trust).
Not a clean monotonic signal; staleness alone doesn't cleanly predict decline past 180
days in this data. Worth keeping in the rule cautiously, not as a strong standalone driver.

**Signal 2 verdict: CONFIRMED.** CTR drops sharply and monotonically as position
worsens: top_3 (1.48) → page_1 (0.65) → striking (0.32) → page_3_5 (0.22) → deep
(0.15), n=2,321 to 7,304 per bucket. Matches Discovery B from Notebook 1.

## 2. Build the ranked queue (writes the CSV)

Encoding the rule from Section 1 and writing the ranked queue to
`work/outputs/baseline_action_score.csv`.

In [3]:
import os

df["is_stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["is_visible"] = (df["impressions_90d"] >= 500).astype(int)
df["score"] = df["is_stale"] * df["is_visible"] * df["impressions_90d"]
df["reason_code"] = "stale_visible_page"
df["action"] = "review_for_refresh"

queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "avg_position", "is_declining_label"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "rows")
queue.head(10)

Wrote 30000 rows


,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,is_declining_label
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,194,61678,19.7,1
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,194,59472,24.8,1
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,194,25715,22.2,1
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,193,13299,10.5,1
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,194,7812,39.0,1
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,193,7558,17.9,1
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,194,4590,31.0,1
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,194,4556,16.4,1
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,194,4429,25.3,1
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,193,1697,15.8,1


## 3. Top-20 review

Top-20 review — action, why, what would make it wrong:

1-17. All `review_for_refresh` / `stale_visible_page`, stale_days 183-301,
impressions_90d 533-61,678. Genuine matches: stale (180+ days) AND visible (500+
impressions). Wrong if: the page was actually refreshed recently and
`days_since_last_update` is stale/unrecorded, or if the traffic is from a single
seasonal spike rather than sustained demand.

18. score=0, stale_days=13, impressions_90d=457 — fails BOTH conditions (too fresh,
below visibility threshold). Only appears because fewer than 20 pages qualify; not a
real candidate. The rule correctly assigns 0.

19. score=0, stale_days=20, impressions_90d=555 — visible but not stale. Correctly
excluded by the rule; not a refresh candidate.

20. score=0, stale_days=104, impressions_90d=43,654 — high traffic, but not stale
enough (104 < 180 days) to trigger the rule. This is the most interesting "miss": a
genuinely high-value page that this narrow rule doesn't flag for anything, since it
isn't stale. Wrong framing if taken as "review_for_refresh" — it isn't; it just
illustrates the rule's blind spot (a page could need a different action entirely,
like expansion, that this single-reason-code rule can't express).

**Finding:** only 17 of 30,000 pages actually satisfy `stale (180+d) AND visible
(500+ impressions)` simultaneously — a much narrower rule than expected. Combined
with Signal 1's MIXED verdict, this suggests staleness alone is a fairly weak,
low-coverage signal in this dataset — worth reconsidering for the Week-5 model to
beat, rather than assuming it's a strong baseline.

In [4]:
top20 = queue.head(20).reset_index(drop=True)
for i, row in top20.iterrows():
    print(f"{i+1}. content_id={row['content_id']} | score={row['score']:.0f} | "
          f"action={row['action']} | reason={row['reason_code']} | "
          f"stale_days={row['days_since_last_update']} | impressions_90d={row['impressions_90d']}")

1. content_id=content_cf56e2e2e282 | score=61678 | action=review_for_refresh | reason=stale_visible_page | stale_days=194 | impressions_90d=61678
2. content_id=content_7368877ea310 | score=59472 | action=review_for_refresh | reason=stale_visible_page | stale_days=194 | impressions_90d=59472
3. content_id=content_1bfaa38ff26c | score=25715 | action=review_for_refresh | reason=stale_visible_page | stale_days=194 | impressions_90d=25715
4. content_id=content_0a91db491d14 | score=13299 | action=review_for_refresh | reason=stale_visible_page | stale_days=193 | impressions_90d=13299
5. content_id=content_5feee3994adb | score=7812 | action=review_for_refresh | reason=stale_visible_page | stale_days=194 | impressions_90d=7812
6. content_id=content_c2d929d83eaa | score=7558 | action=review_for_refresh | reason=stale_visible_page | stale_days=193 | impressions_90d=7558
7. content_id=content_b16bd7307b39 | score=4590 | action=review_for_refresh | reason=stale_visible_page | stale_days=194 | impre

## 4. Weak picks + leakage check

**Weak pick:** row 20 (`content_ccaae106ecb6`, stale_days=104, impressions_90d=43,654) —
technically scores 0 and is correctly excluded, but its presence in a "top 20" list at
all exposes that the rule only has 17 real candidates. A stronger rule would either
lower the staleness threshold or add a second reason code to capture
high-traffic-but-not-stale pages separately.

**Leakage check:** the rule only uses `days_since_last_update` and `impressions_90d`
— both observed, pre-decision signals. `trend_direction`/`trend_pct` (the label
source) were not used anywhere in the score.

In [5]:
# Confirm no label-derived or future columns went into the score
score_inputs = ["is_stale", "is_visible", "impressions_90d"]
leak_risk = [c for c in score_inputs if c in ("trend_direction", "trend_pct")]
print("Leak-risk columns in score inputs:", leak_risk if leak_risk else "None — clean")

Leak-risk columns in score inputs: None — clean


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.